# 🎬 Tormie - Torrent Movie Downloader

Notebook ini akan:
1. Mount Google Drive
2. Install dependencies
3. Mencari film HD dengan seeder terbanyak
4. Mengunduh film via torrent
5. Menyimpan ke Google Drive

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Folder untuk menyimpan film di Google Drive
import os
DOWNLOAD_DIR = '/content/drive/MyDrive/Tormie_Movies'
os.makedirs(DOWNLOAD_DIR, exist_ok=True)
print(f'\u2705 Download folder siap: {DOWNLOAD_DIR}')

## 2. Install Dependencies

In [ ]:
!pip install -q python-libtorrent 2>/dev/null || true
!apt-get install -qq python3-libtorrent -y 2>/dev/null
!pip install -q requests beautifulsoup4 tabulate

## 3. Clone Repository

In [ ]:
# Clone repo tormie (opsional, untuk update script)
!rm -rf /content/tormie
!git clone https://github.com/n0z0/tormie.git /content/tormie
print('\u2705 Repository cloned!')

## 4. Cari Film HD dengan Seeder Terbanyak

In [ ]:
import requests
from bs4 import BeautifulSoup
from tabulate import tabulate
import re


def search_movies_yts(query, quality='720p'):
    """
    Mencari film di YTS API (legal torrent tracker untuk film domain publik).
    Filter berdasarkan kualitas HD dan sort berdasarkan seeds.
    """
    base_url = 'https://yts.mx/api/v2/list_movies.json'
    params = {
        'query_term': query,
        'quality': quality,
        'sort_by': 'seeds',
        'order_by': 'desc',
        'limit': 20
    }
    
    try:
        response = requests.get(base_url, params=params, timeout=10)
        data = response.json()
        
        if data['status'] != 'ok' or data['data']['movie_count'] == 0:
            print('\u274c Tidak ada film ditemukan.')
            return []
        
        movies = data['data']['movies']
        results = []
        
        for movie in movies:
            for torrent in movie.get('torrents', []):
                if torrent['quality'] in ['720p', '1080p', '2160p']:
                    results.append({
                        'title': movie['title'],
                        'year': movie['year'],
                        'rating': movie['rating'],
                        'quality': torrent['quality'],
                        'size': torrent['size'],
                        'seeds': torrent['seeds'],
                        'peers': torrent['peers'],
                        'hash': torrent['hash'],
                        'url': torrent['url']
                    })
        
        # Sort berdasarkan seeds (terbanyak duluan)
        results.sort(key=lambda x: x['seeds'], reverse=True)
        return results
    
    except Exception as e:
        print(f'\u274c Error: {e}')
        return []


def display_results(results):
    """Tampilkan hasil pencarian dalam tabel."""
    if not results:
        return
    
    table_data = []
    for i, r in enumerate(results[:15], 1):
        table_data.append([
            i, r['title'], r['year'], r['rating'],
            r['quality'], r['size'], r['seeds'], r['peers']
        ])
    
    headers = ['#', 'Title', 'Year', 'Rating', 'Quality', 'Size', 'Seeds', 'Peers']
    print(tabulate(table_data, headers=headers, tablefmt='grid'))
    return results

In [ ]:
# --- CARI FILM ---
# Ganti keyword pencarian di bawah ini
SEARCH_QUERY = input('\U0001f50d Masukkan judul film: ') or 'action'
QUALITY = '720p'  # Pilihan: 720p, 1080p, 2160p

print(f'\nMencari "{SEARCH_QUERY}" dengan kualitas {QUALITY}+...\n')
results = search_movies_yts(SEARCH_QUERY, QUALITY)
display_results(results)

## 5. Pilih dan Download Film

In [ ]:
import libtorrent as lt
import time
from IPython.display import clear_output


def download_torrent(torrent_info, save_path):
    """
    Download torrent menggunakan libtorrent.
    Menyimpan langsung ke Google Drive.
    """
    ses = lt.session()
    ses.listen_on(6881, 6891)
    
    # Gunakan magnet link dari hash
    magnet_link = f"magnet:?xt=urn:btih:{torrent_info['hash']}&dn={torrent_info['title']}"
    
    # Tambahkan trackers
    trackers = [
        'udp://open.demonii.com:1337/announce',
        'udp://tracker.openbittorrent.com:80',
        'udp://tracker.coppersurfer.tk:6969',
        'udp://glotorrents.pw:6969/announce',
        'udp://tracker.opentrackr.org:1337/announce',
        'udp://torrent.gresille.org:80/announce',
        'udp://p4p.arenabg.com:1337',
        'udp://tracker.leechers-paradise.org:6969',
    ]
    
    for tracker in trackers:
        magnet_link += f'&tr={tracker}'
    
    params = lt.parse_magnet_uri(magnet_link)
    params.save_path = save_path
    handle = ses.add_torrent(params)
    
    print(f'\U0001f4e5 Downloading: {torrent_info["title"]} [{torrent_info["quality"]}]')
    print(f'\U0001f4c1 Save to: {save_path}')
    print(f'\U0001f331 Seeds: {torrent_info["seeds"]} | Size: {torrent_info["size"]}')
    print('\n--- Progress ---')
    
    while not handle.is_seed():
        s = handle.status()
        
        state_str = ['queued', 'checking', 'downloading metadata',
                     'downloading', 'finished', 'seeding', 'allocating',
                     'checking fastresume']
        
        progress = s.progress * 100
        download_rate = s.download_rate / 1000  # KB/s
        upload_rate = s.upload_rate / 1000
        num_peers = s.num_peers
        
        clear_output(wait=True)
        print(f'\U0001f4e5 Downloading: {torrent_info["title"]} [{torrent_info["quality"]}]')
        print(f'\U0001f4c1 Save to: {save_path}')
        print(f'\n\u2b07\ufe0f  Progress: {progress:.1f}%')
        print(f'\U0001f4ca State: {state_str[s.state]}')
        print(f'\u2b07\ufe0f  Download: {download_rate:.1f} KB/s')
        print(f'\u2b06\ufe0f  Upload: {upload_rate:.1f} KB/s')
        print(f'\U0001f465 Peers: {num_peers}')
        
        # Progress bar
        bar_len = 40
        filled = int(bar_len * s.progress)
        bar = '\u2588' * filled + '\u2591' * (bar_len - filled)
        print(f'\n[{bar}] {progress:.1f}%')
        
        time.sleep(5)
    
    clear_output(wait=True)
    print(f'\u2705 Download selesai: {torrent_info["title"]}')
    print(f'\U0001f4c1 File tersimpan di: {save_path}')
    
    # List downloaded files
    torrent_info_obj = handle.get_torrent_info()
    if torrent_info_obj:
        print(f'\n\U0001f4c4 Files:')
        for i in range(torrent_info_obj.num_files()):
            f = torrent_info_obj.files().file_path(i)
            print(f'   - {f}')
    
    return True

In [ ]:
# --- PILIH FILM UNTUK DIDOWNLOAD ---
if results:
    print('\nHasil pencarian (sorted by seeds):\n')
    display_results(results)
    
    choice = int(input('\n\U0001f3ac Pilih nomor film untuk didownload (1-15): ')) - 1
    
    if 0 <= choice < len(results):
        selected = results[choice]
        print(f'\n\u2705 Dipilih: {selected["title"]} ({selected["year"]}) - {selected["quality"]} - Seeds: {selected["seeds"]}')
        print(f'\U0001f4e6 Size: {selected["size"]}')
        
        confirm = input('\nMulai download? (y/n): ').lower()
        if confirm == 'y':
            download_torrent(selected, DOWNLOAD_DIR)
        else:
            print('\u274c Download dibatalkan.')
    else:
        print('\u274c Pilihan tidak valid.')
else:
    print('\u274c Tidak ada hasil pencarian. Coba keyword lain.')

## 6. Cek File di Google Drive

In [ ]:
import os

print(f'\U0001f4c1 Isi folder {DOWNLOAD_DIR}:\n')
if os.path.exists(DOWNLOAD_DIR):
    for item in os.listdir(DOWNLOAD_DIR):
        item_path = os.path.join(DOWNLOAD_DIR, item)
        if os.path.isfile(item_path):
            size_mb = os.path.getsize(item_path) / (1024 * 1024)
            print(f'   \U0001f4c4 {item} ({size_mb:.1f} MB)')
        elif os.path.isdir(item_path):
            print(f'   \U0001f4c2 {item}/')
else:
    print('   Folder belum ada.')

---
## \u2139\ufe0f Tips
- Pastikan Google Drive memiliki ruang kosong yang cukup
- Film 720p biasanya ~800MB - 1.5GB
- Film 1080p biasanya ~1.5GB - 3GB
- Colab memiliki timeout, jadi pastikan koneksi stabil
- Jika download terputus, jalankan ulang cell download